# rq1: What is the best-performing encoder overall in supervised and SSL settings for HAR tasks?

## Define base configs

In [ ]:
import pandas as pd
metric_used = 'accuracy'
apply_correction_factor = True


def extract_metric_target(df):
    metric_map = {
        "har": "accuracy",
        "hapt": "accuracy",
    }
    return df["pipeline/task"].map(metric_map)


def extract_metric(df):
    metric_results = []
    metric_map = {
        "har": "metric/classification/accuracy",
        "hapt": "metric/classification/accuracy",
    }

    for _, row in df.iterrows():
        metric_column = metric_map[row["pipeline/task"]]
        metric_value = row[metric_column]
        metric_results.append(metric_value)

    return metric_results

In [ ]:
from pathlib import Path


from utils import (
    calculate_variant_wilcoxon,
    create_precedence_graph,
    prepare_experiment_df,
    plot_comparison_bar
)

# disable warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path to the experiment results (parsed) summarized_executions_fixed
filename = 'clean_saved_metrics_paper_final'
summarized_executions_path = Path(f"{filename}.csv")

# summarized_executions_path = Path("summarized_executions_fixed.csv")

# Location to save figures and tables
figures_path = Path("results/figures/")
tables_path = Path("results/tables/")
figures_path.mkdir(parents=True, exist_ok=True)
tables_path.mkdir(parents=True, exist_ok=True)
print(f"Sucessfully created directories '{figures_path}' and '{tables_path}'")

## Utility functions

First, we define some utility functions that will be used to parse the benchmarks results and to generate the plots.

In [ ]:
def result_pairwise_wilcoxon(
    df, variants_variables, filters={}, show_stats=False,apply_bonferroni= False,
):
    df = prepare_experiment_df(
        df,
        **filters,
        verbose=True,
    )
    # display(df)
    if show_stats:
        print("Dataframe has", len(df.index), "rows")
        for col in df.columns:
            values = df[col].unique()
            print(f" - {col} ({len(values)})", values)

    return calculate_variant_wilcoxon(df, variants_variables, threshold=0.05,apply_bonferroni= apply_bonferroni)


def result_precedence_graph(
    df, variants_variables, filters={}, show_stdev=False,apply_bonferroni= False,show_df_tests=False
):
    w_df = result_pairwise_wilcoxon(df, variants_variables, filters,apply_bonferroni= apply_bonferroni,)
    if show_df_tests:
        print("Pairwise Wilcoxon test results:")
        display(w_df)
    return create_precedence_graph(w_df, show_stdev=show_stdev,apply_bonferroni= apply_bonferroni),w_df


def show_precedence_graph(
    df, variants_variables, filters, filename_suffix=None, show_stdev=False,apply_bonferroni= False,show_df_tests=False,filename=None
):

    dot,w_df = result_precedence_graph(
        df=df,
        variants_variables=variants_variables,
        filters=filters,
        show_stdev=show_stdev,
        apply_bonferroni=apply_bonferroni,
        show_df_tests=show_df_tests,
    )
    if filename is None:
        
        filename = "precedence_graph-" + "-".join(variants_variables)
    if filename_suffix:
        filename = filename + "-" + filename_suffix
    dot.render(
        filename=filename, directory=figures_path, format="png", cleanup=True
    )
    print(f"Precedence graph saved to '{figures_path / filename}.png'\n")
    display(dot)
    return dot,w_df


def summarize_backbone_performance(df):
    # Get all unique backbones (from both Variant 1 and Variant 2)
    all_backbones = set(df["Variant 1"]).union(set(df["Variant 2"]))
    
    # Initialize a dictionary to store stats for each backbone
    backbone_stats = {}
    
    for backbone in all_backbones:
        # Get all rows where the backbone appears (either in Variant 1 or Variant 2)
        mask = (df["Variant 1"] == backbone) | (df["Variant 2"] == backbone)
        relevant_rows = df[mask]
        
        # Extract means and stds where the backbone is involved
        means = []
        stds = []
        
        for _, row in relevant_rows.iterrows():
            if row["Variant 1"] == backbone:
                means.append(row["Variant 1 Mean"])
                stds.append(row["Variant 1 stdev"])
            else:
                means.append(row["Variant 2 Mean"])
                stds.append(row["Variant 2 stdev"])
        
        # Compute mean and std across all occurrences
        mean_performance = np.mean(means) if means else 0
        avg_std = np.mean(stds) if stds else 0
        
        backbone_stats[backbone] = {
            "Mean": mean_performance,
            "Std": avg_std,
            "Mean ± Std": f"{np.round(mean_performance*100, 1)}% ± {np.round(avg_std*100, 1)}%"
        }
    # --- Original logic for wins/losses ---
    sig_df = df[df["Significant"] == True].copy()
    
    # Determine winner and loser based on means
    sig_df["Winner"] = sig_df.apply(
        lambda row: row["Variant 1"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 2"], 
        axis=1
    )
    sig_df["Loser"] = sig_df.apply(
        lambda row: row["Variant 2"] if row["Variant 1 Mean"] > row["Variant 2 Mean"] else row["Variant 1"], 
        axis=1
    )
    
    # Count wins and losses
    win_counts = sig_df["Winner"].value_counts()
    loss_counts = sig_df["Loser"].value_counts()
    
    # Create summary DataFrame
    summary_df = pd.DataFrame.from_dict(backbone_stats, orient="index")
    summary_df.index.name = "Backbone"
    summary_df.reset_index(inplace=True)
    
    # Ensure all backbones are included (even if no wins/losses)
    summary_df["Wins"] = summary_df["Backbone"].map(win_counts).fillna(0).astype(int)
    summary_df["Losses"] = summary_df["Backbone"].map(loss_counts).fillna(0).astype(int)
    summary_df["Net Score"] = summary_df["Wins"] - summary_df["Losses"]
    
    # Sort by Net Score (descending)
    summary_df.sort_values(["Net Score", "Mean"], ascending=[False, False], inplace=True)
    
    return summary_df


In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    technique_summary = technique_summary.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    backbone_totals = backbone_totals.sort_values(
        ["Net Score", "Mean"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals



In [ ]:
summarized_executions_path

In [ ]:
df = pd.read_csv(summarized_executions_path)
df

### P1. Qual o melhor backbone geral para HAR?

Qual o melhor backbone para HAR em SSL?
Qual o melhor backbone para HAR em SL?


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ssl_df = df[df["tsk_pretext"] != "Supervised"]
supervised_df = df[df["tsk_pretext"] == "Supervised"]
# Define a custom palette
custom_palette = {
    "RNN": "tab:blue",
    "IMU Transformer": "tab:orange",
    # "ResNet-1D": "tab:green",
    'ResNet-SE-5': 'tab:green',
    "CNN-PFF": "tab:red",
    'TS2Vec Encoder': 'tab:purple',
    'TS-TCC Encoder': 'tab:brown',
    
}
# Define the order of backbones
backbone_order = ["RNN", "IMU Transformer", "ResNet-SE-5","CNN-PFF",'TS2Vec Encoder','TS-TCC Encoder']
# Get colors from Viridis
viridis_colors = sns.color_palette("deep", n_colors=len(backbone_order))

# Create a dictionary mapping backbone names to Viridis colors
viridis_palette = {backbone: color for backbone, color in zip(backbone_order, viridis_colors)}

# Prepare the data as percentage
ssl_df_percent = ssl_df.copy()
supervised_df_percent = supervised_df.copy()
ssl_df_percent["metric"] *= 100
supervised_df_percent["metric"] *= 100

# Determine y-axis limits
ymin = min(ssl_df_percent["metric"].min(), supervised_df_percent["metric"].min())
ymax = max(ssl_df_percent["metric"].max(), supervised_df_percent["metric"].max())
ylims = (max(0, ymin - 5), min(100, ymax + 5))

# Clean aesthetic with larger font
sns.set(style="whitegrid", font_scale=1.5)
custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",
    "grid.color": "black",  # Change grid line color
    "grid.linestyle": "--",  # Change grid line style
    "grid.linewidth": 3  # Change grid line width
}
sns.set_context("paper", rc=custom_params)

label_fontsize = 14
tick_fontsize = 12

# SSL Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x='backbone', y='metric', data=ssl_df_percent, order=backbone_order, palette=viridis_palette)
plt.xlabel("Backbone", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=30, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(ylims)
plt.tight_layout()
plt.savefig('ssl_rq1.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

# Supervised Plot
plt.figure(figsize=(8, 5))
sns.boxplot(x='backbone', y='metric', data=supervised_df_percent, order=backbone_order, palette=viridis_palette)
plt.xlabel("Backbone", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy (%)", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=30, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(ylims)
plt.tight_layout()
plt.savefig('supervised_rq1.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()



## Note:
 use show_df_tests=True, to see intermediate results

In [ ]:
df

In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

# df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('WISDM')]

# df_tnc = df_tnc[df_tnc['backbone'].str.contains('CNN-PFF')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")

In [ ]:
df

In [ ]:
# sanity check
for backbone in df['backbone'].unique():
    df_specific = df[df['backbone'].str.contains(backbone)]
    df_specific = df_specific[df_specific['tsk_pretext']!='Supervised']
    df_specific = df_specific[df_specific['ft_strategy'].str.contains('Full Finetune')]
    mean_accuracy = df_specific['metric'].mean()*100
    std_accuracy = df_specific['metric'].std()*100
    print(f"Task: {backbone} - Mean Accuracy: {mean_accuracy:.1f}, Std Accuracy: {std_accuracy:.1f}")



In [ ]:
df

In [ ]:
# sanity check
for backbone in df['backbone'].unique():
    df_specific = df[df['backbone'].str.contains(backbone)]
    df_specific = df_specific[df_specific['ft_strategy'].str.contains('Freeze')]
    mean_accuracy = df_specific['metric'].mean()*100
    std_accuracy = df_specific['metric'].std()*100
    print(f"Task: {backbone} - Mean Accuracy: {mean_accuracy:.1f}, Std Accuracy: {std_accuracy:.1f}")



In [ ]:
df

In [ ]:
# sanity check
for tsk_pretext in df['tsk_pretext'].unique():
    df_specific = df[df['tsk_pretext'].str.contains(tsk_pretext)]
    df_specific = df_specific[df_specific['ft_strategy'].str.contains('Full Finetune')]
    mean_accuracy = df_specific['metric'].mean()*100
    std_accuracy = df_specific['metric'].std()*100
    print(f"Task: {tsk_pretext} - Mean Accuracy: {mean_accuracy:.1f}, Std Accuracy: {std_accuracy:.1f}")



In [ ]:
# sanity check

df_tnc = df[df['tsk_pretext'].str.contains('TNC')]
df_tnc

df_tnc = df_tnc[df_tnc['ft_strategy'].str.contains('Full Finetune')]

# df_tnc = df_tnc[df_tnc['d_pretext'].str.contains('WISDM')]

df_tnc = df_tnc[df_tnc['backbone'].str.contains('CNN-PFF')]

display(df_tnc)

# mean and std of sccuracy
mean_accuracy = df_tnc['metric'].mean()*100
std_accuracy = df_tnc['metric'].std()*100
print(f"Mean Accuracy: {mean_accuracy:.1f}")
print(f"Std Accuracy: {std_accuracy:.1f}")

In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        # "select_tsk_pretext": [ "TNC","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_all_rq1"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] #+ ' + SSL'

summary_df_ssl_rq12


In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        # "select_tsk_pretext": [ "TNC","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_all_rq1"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] #+ ' + SSL'

summary_df_ssl_rq12


In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        "select_tsk_pretext": [ "TNC","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_ssl_rq1"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

summary_df_ssl_rq12




In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        "select_tsk_pretext": [ "TNC","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_ssl_rq1_fullfinetune"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

summary_df_ssl_rq12




In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        "select_tsk_pretext": [ "TNC","Diet","TFC","LFR"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_ssl_rq1_freeze"
    
)




summary_df_ssl_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_ssl_rq12)
summary_df_ssl_rq12['Backbone'] = summary_df_ssl_rq12['Backbone'] + ' + SSL'

summary_df_ssl_rq12




In [ ]:
import numpy as np
# SSL without ts2vec - will be definitive
dot,df_ssl_rq12 = show_precedence_graph(
    df,
    variants_variables=["backbone"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune",],    # Only use the full finetuning strategy
        # "select_backbones": ["ResNet","RNN","Transformer","CNN"],   # "TS2Vec",  Only use these backbones "TS2Vec",
        "select_tsk_pretext": [ "Supervised"],  # ,"Supervised" only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_sl_rq1"
    
)




summary_df_supervised_rq12 = summarize_backbone_performance(df_ssl_rq12)
display(summary_df_supervised_rq12)
summary_df_supervised_rq12['Backbone'] = summary_df_supervised_rq12['Backbone'] + ' + Supervised'

summary_df_supervised_rq12




In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Filter relevant columns
df_filtered = df[["backbone", "tsk_pretext", "metric"]]

# Aggregate mean metric values per backbone and task
df_grouped = df_filtered.groupby(["backbone", "tsk_pretext"], as_index=False).mean()
display(df_grouped)
# Rename columns
df_grouped.rename(columns={"metric": "Score"}, inplace=True)

# Sort the dataframe by Score within each tsk_pretext
df_grouped = df_grouped.sort_values(by=["tsk_pretext", "Score"], ascending=[True, False])
# df_grouped["Accuracy"] *= 100

# Plot
# Create the plot
plt.figure(figsize=(10, 6))
ax = sns.barplot(x="tsk_pretext", y="Score", hue="backbone", data=df_grouped, 
                 palette=viridis_palette, errwidth=1.5, capsize=0.05)


plt.xlabel("Task Pretext", fontsize=label_fontsize, fontweight='bold')
plt.ylabel("Balanced Accuracy", fontsize=label_fontsize, fontweight='bold')
plt.xticks(rotation=45, fontsize=tick_fontsize, fontweight='bold')
plt.yticks(fontsize=tick_fontsize, fontweight='bold')
plt.ylim(0, 1)

# Legend styling
legend = plt.legend(title="Backbone", bbox_to_anchor=(1, 1), 
                   fontsize=12)
plt.setp(legend.get_title(), fontsize=12, fontweight='bold')

# Adjust layout and save
plt.tight_layout()
plt.savefig('ssl_sl_rq2.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:


finetune_df = df[df["ft_strategy"] == "Full Finetune"]
freeze_df = df[df["ft_strategy"] == "Freeze"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define a custom palette
custom_palette = {
    "RNN": "blue",
    "IMU Transformer": "red",
    "ResNet-SE-5": "green",
    "CNN-PFF": "orange",
    'TS2Vec Encoder': 'tab:purple',
    'TS-TCC Encoder': 'tab:brown',
    # 'ResNet-SE-5': 'tab:pink',
}



# Convert frac_dtarget to categorical and update to percentage strings
# try:
#     finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(float) * 100
#     # finetune_df["frac_dtarget"] = finetune_df["frac_dtarget"].astype(str) + '%'
# except ValueError as e:
#     print(f"Error converting frac_dtarget: {e}")

# Group by backbone and frac_dtarget, and calculate mean accuracy and standard deviation
mean_df = finetune_df.groupby(['backbone','frac_dtarget'])['metric'].agg(['mean', 'std']).reset_index()
display(mean_df)
mean_df["frac_dtarget"] = mean_df["frac_dtarget"].astype(str)
# Set enhanced style parameters
sns.set_style("whitegrid", rc={
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.labelweight": "bold",  # Bold axis labels
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.titlesize": 14,        # Larger title (though we'll remove it)
})

# Create figure with larger dimensions
plt.figure(figsize=(11, 7), layout='constrained')

# Create pointplot with enhanced visibility
ax = sns.pointplot(
    data=mean_df, 
    x='frac_dtarget', 
    y='mean', 
    hue='backbone',
    # palette=viridis_palette,
    hue_order=["RNN", "IMU Transformer", "ResNet-SE-5", "CNN-PFF", "TS2Vec Encoder",'TS-TCC Encoder'],
    order=["1.0", "5.0", "10.0", "50.0", "100.0","200.0","1000.0"],
    errwidth=2.0,      # Thicker error bars
    capsize=0.15,      # Slightly larger caps
    # markersize=10      # Larger points
)

# Enhanced axis labels
plt.xlabel("Data Percentage", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)
plt.ylabel("Balanced Accuracy", 
          fontsize=16,  # Increased from 12
          fontweight='bold',
          labelpad=12)

# Bold axis ticks with larger font
plt.xticks(rotation=45, 
          fontsize=13,  # Increased from 11
          fontweight='bold')
plt.yticks(fontsize=13,  # Increased from 11
          fontweight='bold')

# Y-axis formatting
plt.ylim(0.3, 1.0)
ax.set_yticklabels(['{:.0f}%'.format(y*100) for y in ax.get_yticks()])

# Enhanced grid
plt.grid(True, alpha=0.7)

# Upgraded legend
handles, labels = ax.get_legend_handles_labels()
legend = plt.legend(
    handles, 
    labels, 
    title="Backbone",
    title_fontsize=18,  # Increased from 12
    fontsize=16,        # Increased from 11
    bbox_to_anchor=(1, 1),
    loc='upper left',
    frameon=True,
    framealpha=1,
    edgecolor='black'
)

# Make legend title bold
legend.get_title().set_fontweight('bold')

# Save high-quality output
plt.savefig('finetune_rq4.png', 
           dpi=350,      # Higher than standard 300
           bbox_inches='tight', 
           transparent=True)

plt.show()

In [ ]:
# lfr commented

In [ ]:
# all backbones for TNC
dot,df_tnc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TNC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tnc_rq2"
    
)

df_tnc

summary_df_tnc = summarize_backbone_performance(df_tnc)
display(summary_df_tnc)
# all backbones for TFC
dot,df_tfc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TFC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tfc_rq2"
    
)

df_tfc

summary_df_tfc = summarize_backbone_performance(df_tfc)
display(summary_df_tfc)
# all backbones for DIET
dot,df_diet = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["Diet"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_diet_rq2"
    
)

df_diet

summary_df_diet = summarize_backbone_performance(df_diet)
display(summary_df_diet)
# all backbones for LFR
dot,df_lfr = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        # "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["LFR"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_lfr_rq2"
    
)

df_lfr

summary_df_lfr = summarize_backbone_performance(df_lfr)
display(summary_df_lfr)
# Combine all DataFrames
combined_df_rq2 = pd.concat([
    summary_df_supervised_rq12,
    summary_df_tnc,
    summary_df_tfc,
    summary_df_diet,
    summary_df_lfr
], ignore_index=False)

# Compute total wins/losses per backbone across all techniques
summary_df_rq2 = combined_df_rq2.groupby("Backbone").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Mean": "mean",    # Average mean across all techniques
    "Std": "mean"      # Average std across all techniques
}).reset_index()

# Calculate Net Score
summary_df_rq2["Net Score"] = summary_df_rq2["Wins"] - summary_df_rq2["Losses"]
summary_df_rq2["Mean"] = summary_df_rq2["Mean"] * 100
summary_df_rq2["Std"] = summary_df_rq2["Std"] * 100
# Format Mean ± Std (e.g., "75.9 ± 1.3")
summary_df_rq2["Performance"] = (
    summary_df_rq2["Mean"].round(1).astype(str) + 
    "% ± " + 
    summary_df_rq2["Std"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
# summary_df_rq2.sort_values(
#     ["Net Score", "Mean"], 
#     ascending=[False, False], 
#     inplace=True
# )

summary_df_rq2.sort_values(
    ["Mean","Net Score"], 
    ascending=[False, False], 
    inplace=True
)

# Reorder columns for clarity
summary_df_rq2 = summary_df_rq2[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(summary_df_rq2)

# Split the backbone name and create new column
summary_df_rq2["Backbone Only"] = summary_df_rq2["Backbone"].str.split(" \+ ").str[0]

# Extract numeric values from Performance column for calculations
summary_df_rq2[['Mean_pct', 'Std_pct']] = summary_df_rq2['Performance'].str.extract(r'(\d+\.\d+)% ± (\d+\.\d+)%').astype(float)

# Group by Backbone and aggregate all metrics
backbone_totals = summary_df_rq2.groupby("Backbone Only").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Net Score": "sum",
    "Mean_pct": "mean",  # Average mean across all variants
    "Std_pct": "mean"    # Average std across all variants
}).reset_index()

# Format Performance column with percentage
backbone_totals["Performance"] = (
    backbone_totals["Mean_pct"].round(1).astype(str) + 
    "% ± " + 
    backbone_totals["Std_pct"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
# backbone_totals = backbone_totals.sort_values(
#     ["Net Score", "Mean_pct"], 
#     ascending=[False, False]
# )
backbone_totals = backbone_totals.sort_values(
    ["Mean_pct","Net Score"], 
    ascending=[False, False]
)

# Rename and clean up columns
backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
backbone_totals.replace({"Backbone": {
    'TS2Vec': 'TS2Vec (Partial)',
}}, inplace=True)

# Select final columns
backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(backbone_totals)



In [ ]:
# all backbones for TNC
dot,df_tnc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TNC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tnc_rq2"
    
)

df_tnc

summary_df_tnc = summarize_backbone_performance(df_tnc)
display(summary_df_tnc)
# all backbones for TFC
dot,df_tfc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TFC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tfc_rq2"
    
)

df_tfc

summary_df_tfc = summarize_backbone_performance(df_tfc)
display(summary_df_tfc)
# all backbones for DIET
dot,df_diet = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["Diet"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_diet_rq2"
    
)

df_diet

summary_df_diet = summarize_backbone_performance(df_diet)
display(summary_df_diet)
# all backbones for LFR
dot,df_lfr = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Full Finetune"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["LFR"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_lfr_rq2"
    
)

df_lfr

summary_df_lfr = summarize_backbone_performance(df_lfr)
display(summary_df_lfr)
# Combine all DataFrames
combined_df_rq2 = pd.concat([
    # summary_df_supervised_rq12,
    summary_df_tnc,
    summary_df_tfc,
    summary_df_diet,
    summary_df_lfr
], ignore_index=False)

# Compute total wins/losses per backbone across all techniques
summary_df_rq2 = combined_df_rq2.groupby("Backbone").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Mean": "mean",    # Average mean across all techniques
    "Std": "mean"      # Average std across all techniques
}).reset_index()

# Calculate Net Score
summary_df_rq2["Net Score"] = summary_df_rq2["Wins"] - summary_df_rq2["Losses"]
summary_df_rq2["Mean"] = summary_df_rq2["Mean"] * 100
summary_df_rq2["Std"] = summary_df_rq2["Std"] * 100
# Format Mean ± Std (e.g., "75.9 ± 1.3")
summary_df_rq2["Performance"] = (
    summary_df_rq2["Mean"].round(1).astype(str) + 
    "% ± " + 
    summary_df_rq2["Std"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
# summary_df_rq2.sort_values(
#     ["Net Score", "Mean"], 
#     ascending=[False, False], 
#     inplace=True
# )

summary_df_rq2.sort_values(
    ["Mean","Net Score"], 
    ascending=[False, False], 
    inplace=True
)

# Reorder columns for clarity
summary_df_rq2 = summary_df_rq2[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(summary_df_rq2)

# Split the backbone name and create new column
summary_df_rq2["Backbone Only"] = summary_df_rq2["Backbone"].str.split(" \+ ").str[0]

# Extract numeric values from Performance column for calculations
summary_df_rq2[['Mean_pct', 'Std_pct']] = summary_df_rq2['Performance'].str.extract(r'(\d+\.\d+)% ± (\d+\.\d+)%').astype(float)

# Group by Backbone and aggregate all metrics
backbone_totals = summary_df_rq2.groupby("Backbone Only").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Net Score": "sum",
    "Mean_pct": "mean",  # Average mean across all variants
    "Std_pct": "mean"    # Average std across all variants
}).reset_index()

# Format Performance column with percentage
backbone_totals["Performance"] = (
    backbone_totals["Mean_pct"].round(1).astype(str) + 
    "% ± " + 
    backbone_totals["Std_pct"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
# backbone_totals = backbone_totals.sort_values(
#     ["Net Score", "Mean_pct"], 
#     ascending=[False, False]
# )
backbone_totals = backbone_totals.sort_values(
    ["Mean_pct","Net Score"], 
    ascending=[False, False]
)

# Rename and clean up columns
backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
backbone_totals.replace({"Backbone": {
    'TS2Vec': 'TS2Vec (Partial)',
}}, inplace=True)

# Select final columns
backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(backbone_totals)



In [ ]:
# all backbones for TNC
dot,df_tnc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TNC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tnc_rq2"
    
)

df_tnc

summary_df_tnc = summarize_backbone_performance(df_tnc)
display(summary_df_tnc)
# all backbones for TFC
dot,df_tfc = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["TFC"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_tfc_rq2"
    
)

df_tfc

summary_df_tfc = summarize_backbone_performance(df_tfc)
display(summary_df_tfc)
# all backbones for DIET
dot,df_diet = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["Diet"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_diet_rq2"
    
)

df_diet

summary_df_diet = summarize_backbone_performance(df_diet)
display(summary_df_diet)
# all backbones for LFR
dot,df_lfr = show_precedence_graph(
    df,
    variants_variables=["backbone","tsk_pretext"],                # Name of the columns to compare (variant)
    filters={
        # "select_frac_dtarget": [1.0],               # Only use the full dataset
        "select_ft_strategy": ["Freeze"],    # Only use the full finetuning strategy
        # "select_backbones": ["TS2Vec", "ResNet","RNN","Transformer","CNN"],   # Only use these backbones
        "select_tsk_pretext": ["LFR"],  # Only use these pretext tasks
    },
    show_stdev=True,
    apply_bonferroni= apply_correction_factor,
    filename = "wilcoxon_lfr_rq2"
    
)

df_lfr

summary_df_lfr = summarize_backbone_performance(df_lfr)
display(summary_df_lfr)
# Combine all DataFrames
combined_df_rq2 = pd.concat([
    # summary_df_supervised_rq12,
    summary_df_tnc,
    summary_df_tfc,
    summary_df_diet,
    summary_df_lfr
], ignore_index=False)

# Compute total wins/losses per backbone across all techniques
summary_df_rq2 = combined_df_rq2.groupby("Backbone").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Mean": "mean",    # Average mean across all techniques
    "Std": "mean"      # Average std across all techniques
}).reset_index()

# Calculate Net Score
summary_df_rq2["Net Score"] = summary_df_rq2["Wins"] - summary_df_rq2["Losses"]
summary_df_rq2["Mean"] = summary_df_rq2["Mean"] * 100
summary_df_rq2["Std"] = summary_df_rq2["Std"] * 100
# Format Mean ± Std (e.g., "75.9 ± 1.3")
summary_df_rq2["Performance"] = (
    summary_df_rq2["Mean"].round(1).astype(str) + 
    "% ± " + 
    summary_df_rq2["Std"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
# summary_df_rq2.sort_values(
#     ["Net Score", "Mean"], 
#     ascending=[False, False], 
#     inplace=True
# )

summary_df_rq2.sort_values(
    ["Mean","Net Score"], 
    ascending=[False, False], 
    inplace=True
)

# Reorder columns for clarity
summary_df_rq2 = summary_df_rq2[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(summary_df_rq2)

# Split the backbone name and create new column
summary_df_rq2["Backbone Only"] = summary_df_rq2["Backbone"].str.split(" \+ ").str[0]

# Extract numeric values from Performance column for calculations
summary_df_rq2[['Mean_pct', 'Std_pct']] = summary_df_rq2['Performance'].str.extract(r'(\d+\.\d+)% ± (\d+\.\d+)%').astype(float)

# Group by Backbone and aggregate all metrics
backbone_totals = summary_df_rq2.groupby("Backbone Only").agg({
    "Wins": "sum",
    "Losses": "sum",
    "Net Score": "sum",
    "Mean_pct": "mean",  # Average mean across all variants
    "Std_pct": "mean"    # Average std across all variants
}).reset_index()

# Format Performance column with percentage
backbone_totals["Performance"] = (
    backbone_totals["Mean_pct"].round(1).astype(str) + 
    "% ± " + 
    backbone_totals["Std_pct"].round(1).astype(str) + "%"
)

# Sort by Net Score (descending), then by Mean (descending) for ties
# backbone_totals = backbone_totals.sort_values(
#     ["Net Score", "Mean_pct"], 
#     ascending=[False, False]
# )
backbone_totals = backbone_totals.sort_values(
    ["Mean_pct","Net Score"], 
    ascending=[False, False]
)

# Rename and clean up columns
backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
backbone_totals.replace({"Backbone": {
    'TS2Vec': 'TS2Vec (Partial)',
}}, inplace=True)

# Select final columns
backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]

display(backbone_totals)



In [ ]:
def aggregate_backbone_performance(combined_df):
    """
    Processes a combined DataFrame of backbone results to produce:
    1. Technique-specific performance (mean ± std)
    2. Backbone totals across all techniques
    
    Args:
        combined_df: DataFrame containing results from multiple techniques
        
    Returns:
        tuple: (technique_summary_df, backbone_totals_df)
    """
    # --- Technique-Specific Summary ---
    technique_summary = combined_df.copy()
    
    # Convert to percentages if needed (assuming original means are 0-1)
    technique_summary["Mean"] = technique_summary["Mean"] * 100
    technique_summary["Std"] = technique_summary["Std"] * 100
    
    # Format performance string
    technique_summary["Performance"] = (
        technique_summary["Mean"].round(1).astype(str) + 
        "% ± " + 
        technique_summary["Std"].round(1).astype(str) + "%"
    )
    
    # Sort by Net Score then Mean
    # technique_summary = technique_summary.sort_values(
    #     ["Net Score", "Mean"], 
    #     ascending=[False, False]
    # )
    technique_summary = technique_summary.sort_values(
        ["Mean","Net Score"], 
        ascending=[False, False]
    )
    # --- Backbone Totals ---
    # Extract backbone name (before " + ")
    combined_df["Backbone Only"] = combined_df["Backbone"].str.split(" \+ ").str[0]
    
    # Group and aggregate
    backbone_totals = combined_df.groupby("Backbone Only").agg({
        "Wins": "sum",
        "Losses": "sum",
        "Net Score": "sum",
        "Mean": lambda x: np.mean(x) * 100,  # Convert to percentage
        "Std": lambda x: np.mean(x) * 100
    }).reset_index()
    
    # Format performance
    backbone_totals["Performance"] = (
        backbone_totals["Mean"].round(1).astype(str) + 
        "% ± " + 
        backbone_totals["Std"].round(1).astype(str) + "%"
    )
    
    # Clean up
    backbone_totals = backbone_totals.rename(columns={"Backbone Only": "Backbone"})
    # backbone_totals = backbone_totals.sort_values(
    #     ["Net Score", "Mean"], 
    #     ascending=[False, False]
    # )
    backbone_totals = backbone_totals.sort_values(
        [ "Mean","Net Score"], 
        ascending=[False, False]
    )
    
    # Special handling for TS2Vec if present
    if "TS2Vec" in backbone_totals["Backbone"].values:
        backbone_totals["Backbone"] = backbone_totals["Backbone"].replace({
            "TS2Vec": "TS2Vec (Partial)"
        })
    
    # Select final columns
    backbone_totals = backbone_totals[["Backbone", "Performance", "Wins", "Losses", "Net Score"]]
    
    return technique_summary, backbone_totals

def generate_performance_tables(df_list, technique_names=None):
    """
    Complete workflow from individual technique DataFrames to final tables
    
    Args:
        df_list: List of DataFrames for each technique
        technique_names: Optional list of technique names
        
    Returns:
        tuple: (combined_df, technique_summary, backbone_totals)
    """
    # Add technique identifiers if provided
    if technique_names and len(technique_names) == len(df_list):
        for df, name in zip(df_list, technique_names):
            df["Technique"] = name
    
    # Combine all DataFrames
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Generate summary tables
    technique_summary, backbone_totals = aggregate_backbone_performance(combined_df)
    
    return combined_df, technique_summary, backbone_totals

# Assuming you have these individual technique DataFrames:
technique_dfs = [
    summary_df_tnc,
    summary_df_tfc,
    summary_df_diet,
    summary_df_lfr
]

technique_names = [
    "TNC",
    "TFC",
    "Diet",
    "LFR"
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)

In [ ]:
# all combined
technique_dfs = [
# summary_df_ssl_rq12,
summary_df_supervised_rq12,
summary_df_tnc,
summary_df_tfc,
summary_df_diet,
summary_df_lfr
]

technique_names = [
    "SSL",
    "SSL with TS2Vec",
    "Supervised"
    "TNC",
    "TFC",
    "Diet",
    "LFR"
]

# Generate all tables
combined_df, technique_summary, backbone_totals = generate_performance_tables(
    technique_dfs, 
    technique_names
)

# Display results
print("=== Technique-Specific Performance ===")
display(technique_summary)

print("\n=== Backbone Totals Across All Techniques ===")
display(backbone_totals)